In [ ]:
# 1) Install (takes ~1 min). If Colab asks, restart the session afterwards.
!apt-get -qq install -y espeak-ng > /dev/null
!pip -q install --no-deps kokoro misaki
!pip -q install soundfile loguru num2words addict phonemizer-fork espeakng-loader
!python -m spacy download en_core_web_sm

In [ ]:
# 2) Settings + connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

STORIES_DIR = '/content/drive/MyDrive/Audio Generation/stories'
OUTPUT_DIR  = '/content/drive/MyDrive/Audio Generation/audio'
VOICE = 'af_bella'
BITRATE = '64k'    # MP3 quality for speech: '48k' = smaller, '96k' = higher
FORCE = False      # True = regenerate even if audio exists
LIMIT = 2          # test with 2 stories first; set to None to do them all

import os, torch
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('GPU available:', torch.cuda.is_available())

In [ ]:
# 3) Helpers (Python port of story-parser.js, chunker.js, wav.js)
import re, numpy as np

SR = 24000

# ---------- parser ----------
KEY_LINE = re.compile(r'^\s*(title|description|url)\s*:\s?(.*)$', re.I)
SEPARATOR = re.compile(r'^\s*={3,}\s*$')

def read_text(path):
    b = open(path, 'rb').read()
    if b[:2] == b'\xff\xfe': return b[2:].decode('utf-16-le')
    if b[:2] == b'\xfe\xff': return b[2:].decode('utf-16-be')
    return b.decode('utf-8-sig', errors='replace')

def parse_story(raw, file_name=''):
    text = raw.lstrip('\ufeff').replace('\r\n', '\n').replace('\r', '\n')
    lines = text.split('\n')
    sep = -1
    for i in range(min(len(lines), 80)):
        if SEPARATOR.match(lines[i]):
            sep = i
            break
    body_lines = lines
    header_lines = []
    if sep >= 0:
        cand = lines[:sep]
        if any(KEY_LINE.match(l) for l in cand):
            header_lines = cand
            body_lines = lines[sep + 1:]
    else:
        first = next((i for i, l in enumerate(lines) if l.strip()), -1)
        if first >= 0 and KEY_LINE.match(lines[first]):
            end = first
            while end < len(lines) and lines[end].strip() != '':
                end += 1
            header_lines = lines[first:end]
            body_lines = lines[end:]
    title = ''
    for l in header_lines:
        m = KEY_LINE.match(l)
        if m and m.group(1).lower() == 'title':
            title = m.group(2).strip()
    body = '\n'.join(body_lines).lstrip('\n').rstrip()
    return {'title': title or os.path.splitext(file_name)[0], 'text': body}

# ---------- chunker ----------
MAX_CHARS, FIRST_MAX_CHARS = 280, 140
PAUSE = {'sentence': 0.08, 'line': 0.4, 'paragraph': 0.6, 'sceneBreak': 0.9}
ABBR = set('mr mrs ms dr prof sr jr st mt vs etc fig gen col lt sgt capt rev hon inc ltd co jan feb mar apr jun jul aug sep sept oct nov dec e.g i.e a.m p.m'.split())
TERMINATORS = '.!?…'
CLOSERS = '"\'”’)]»'
SCENE_BREAK = re.compile(r'^[\s*\-_=~#•·.+\u2013\u2014]{3,}$')
ENDS_SENTENCE = re.compile(r'[.!?…:;"\'”’)\]*_]$')

def normalize(s):
    s = re.sub(r'\r\n?|\u2028|\u2029', '\n', s)
    s = re.sub(r'[\u200B-\u200D\uFEFF\u2060]', '', s)
    return s.replace('\u00A0', ' ').replace('\t', ' ')

def is_abbreviation(text, pi):
    k = pi - 1
    while k >= 0 and re.match(r'[A-Za-z.]', text[k]):
        k -= 1
    token = text[k + 1:pi]
    if not token: return False
    if token.lower() in ABBR: return True
    return bool(re.fullmatch(r'([A-Za-z]\.)*[A-Za-z]', token))

def split_sentences(text):
    out, n, start, i = [], len(text), 0, 0
    while i < n:
        if text[i] not in TERMINATORS:
            i += 1
            continue
        first, end = text[i], i
        while end + 1 < n and text[end + 1] in TERMINATORS: end += 1
        while end + 1 < n and text[end + 1] in CLOSERS: end += 1
        if end + 1 < n and not text[end + 1].isspace():
            i = end + 1
            continue
        j = end + 1
        while j < n and text[j].isspace(): j += 1
        if j < n:
            if text[j].islower():
                i = end + 1
                continue
            if first == '.' and end == i and is_abbreviation(text, i):
                i = end + 1
                continue
        out.append(text[start:end + 1])
        start = j
        i = j
    if start < n: out.append(text[start:])
    return [s.strip() for s in out if s.strip()]

def split_at_spaces(s, mx):
    pieces, buf = [], ''
    for word in s.split():
        if len(word) > mx:
            if buf: pieces.append(buf)
            buf = ''
            pieces += [word[p:p + mx] for p in range(0, len(word), mx)]
            continue
        if buf and len(buf) + 1 + len(word) > mx:
            pieces.append(buf)
            buf = word
        else:
            buf = f'{buf} {word}' if buf else word
    if buf: pieces.append(buf)
    return pieces

def split_long(s, mx):
    pieces, buf = [], ''
    for clause in re.split(r'(?<=[,;:\u2013\u2014])\s+', s):
        if len(clause) > mx:
            if buf: pieces.append(buf)
            buf = ''
            pieces += split_at_spaces(clause, mx)
        elif buf and len(buf) + 1 + len(clause) > mx:
            pieces.append(buf)
            buf = clause
        else:
            buf = f'{buf} {clause}' if buf else clause
    if buf: pieces.append(buf)
    return pieces

def to_paragraphs(body):
    paras, cur = [], None
    def flush(gap):
        nonlocal cur
        if cur is not None: paras.append({'text': cur, 'gap': gap})
        cur = None
    for raw in normalize(body).split('\n'):
        line = raw.strip()
        if line == '':
            flush(2)
        elif SCENE_BREAK.match(line):
            flush(2)
            paras.append({'scene': True})
        elif cur is None:
            cur = line
        elif not ENDS_SENTENCE.search(cur) and line[0].islower():
            cur += ' ' + line
        else:
            flush(1)
            cur = line
    flush(2)
    return paras

def chunk_text(body):
    chunks = []
    def push(t):
        t = re.sub(r'\s+', ' ', t).strip()
        if any(ch.isalnum() for ch in t):
            chunks.append({'text': t, 'pause': PAUSE['sentence']})
    for para in to_paragraphs(body):
        if para.get('scene'):
            if chunks: chunks[-1]['pause'] += PAUSE['sceneBreak']
            continue
        before, buf = len(chunks), ''
        for sentence in split_sentences(para['text']):
            cap = FIRST_MAX_CHARS if len(chunks) == 0 else MAX_CHARS
            if len(sentence) > cap:
                if buf: push(buf)
                buf = ''
                for piece in split_long(sentence, cap): push(piece)
            elif buf and len(buf) + 1 + len(sentence) > cap:
                push(buf)
                buf = sentence
            else:
                buf = f'{buf} {sentence}' if buf else sentence
        if buf: push(buf)
        if len(chunks) > before:
            chunks[-1]['pause'] = PAUSE['paragraph'] if para['gap'] >= 2 else PAUSE['line']
    return chunks

# ---------- audio helpers ----------
def clean_for_tts(t):
    t = re.sub(r'([a-zA-Z])\1{2,}', r'\1\1', t)
    t = re.sub(r'(\w+)\.\.\.(\w+)', r'\1, \2', t)
    return t.replace('...', '—')

def trim_silence(x, threshold=0.008, lead=0.04, tail=0.06):
    idx = np.where(np.abs(x) > threshold)[0]
    if len(idx) == 0: return x[:int(SR * 0.1)]
    s = max(0, idx[0] - int(lead * SR))
    e = min(len(x), idx[-1] + 1 + int(tail * SR))
    return x[s:e]

print('Helpers ready')

In [ ]:
# 4) Generate audio as MP3 (safe to re-run: finished stories are skipped)
import time, subprocess, logging
logging.getLogger('phonemizer').setLevel(logging.ERROR)
from kokoro import KPipeline

def save_mp3(audio, path, bitrate):
    tmp = path + '.tmp'   # write to a temp name so a cutoff never leaves a broken file
    p = subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-f', 'f32le', '-ar', str(SR), '-ac', '1',
         '-i', 'pipe:0', '-codec:a', 'libmp3lame', '-b:a', bitrate, '-f', 'mp3', tmp],
        input=audio.astype('<f4').tobytes(), capture_output=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr.decode()[:300])
    os.replace(tmp, path)

pipeline = KPipeline(lang_code='a')   # 'a' = American English (af_bella)

files = sorted(f for f in os.listdir(STORIES_DIR) if f.lower().endswith('.txt'))
todo = []
for f in files:
    base = os.path.splitext(f)[0]
    wav = os.path.join(OUTPUT_DIR, base + '.wav')
    mp3 = os.path.join(OUTPUT_DIR, base + '.mp3')
    if (os.path.exists(wav) or os.path.exists(mp3)) and not FORCE:
        continue
    todo.append((f, mp3))
print(f'{len(files)} stories found, {len(todo)} to generate')
if LIMIT: todo = todo[:LIMIT]

for n, (f, mp3) in enumerate(todo, 1):
    try:
        parsed = parse_story(read_text(os.path.join(STORIES_DIR, f)), f)
        body = parsed['text'].strip()
        if not body:
            print(f'[{n}/{len(todo)}] {f}: no body text, skipping')
            continue
        chunks = chunk_text(body)
        t0, parts = time.time(), []
        for c in chunks:
            segs = []
            for _, _, a in pipeline(clean_for_tts(c['text']), voice=VOICE):
                if a is not None:
                    segs.append(a.detach().cpu().numpy() if hasattr(a, 'detach') else np.asarray(a))
            if not segs: continue
            parts.append(trim_silence(np.concatenate(segs).astype(np.float32)))
            parts.append(np.zeros(round(c['pause'] * SR), dtype=np.float32))
        if not parts:
            print(f'[{n}/{len(todo)}] {f}: no audio produced')
            continue
        audio = np.concatenate(parts)
        save_mp3(audio, mp3, BITRATE)
        print(f'[{n}/{len(todo)}] {f}: {len(audio)/SR/60:.1f} min of audio in {time.time()-t0:.0f}s')
    except Exception as e:
        print(f'[{n}/{len(todo)}] {f}: ERROR {e}')
print('Done')